# Day 6 & 7 Session Summary
## Dark Store Placement + Integrated Forward–Reverse Logistics

**Branch:** `pritam_temp_apr5`  
**Commits:** `69b5a8b` (Day 6) → `70d7acd` (Day 7) → `9766c07` (Day 7 bug fix)  
**Session focus:** Day 7 — LaTeX/PDF report + `run_all.sh` + Pareto sweep bug fix

---

### Session Objectives

| # | Day | Objective | Status | Commit |
|---|-----|-----------|--------|--------|
| 1 | 6 | `src/kpi_reporter.py` — combined KPI report + zone priority ranking | ✅ Complete | `69b5a8b` |
| 2 | 6 | `pareto_sweep()` in `src/joint_optimizer.py` | ✅ Complete | `69b5a8b` |
| 3 | 6 | `notebooks/06_kpi_and_pareto.ipynb` — 12-cell notebook | ✅ Complete | `69b5a8b` |
| 4 | 7 | `src/report_builder.py` — LaTeX/PDF report in `report/` | ✅ Complete | `70d7acd` |
| 5 | 7 | `run_all.sh` — one-shot reproducibility script | ✅ Complete | `70d7acd` |
| 6 | 7 | Session summary notebook | ✅ Complete | `70d7acd` |
| 7 | 7 | **Bug fix:** Pareto sweep single-point issue (3 root causes) | ✅ Fixed | `9766c07` |
| 8 | 7 | Report regenerated with corrected Pareto data | ✅ Done | (this session) |


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, '.')

import pandas as pd
from IPython.display import display

tasks = [
    # Day 6 tasks
    {"Task ID": "D6-1", "Description": "src/kpi_reporter.py — load_forward/reverse/hybrid, build_combined, zone_priority_ranking", "Status": "✅ Completed", "Commit": "69b5a8b", "Notes": "R$4,873.91 sep total; Zone 8 rank 1"},
    {"Task ID": "D6-2", "Description": "pareto_sweep() added to src/joint_optimizer.py (5×5 grid, non-dominance, knee point)", "Status": "✅ Completed", "Commit": "69b5a8b", "Notes": "15 valid combos; outputs/pareto_results.csv"},
    {"Task ID": "D6-3", "Description": "notebooks/06_kpi_and_pareto.ipynb — 12 cells (theory + calls + charts)", "Status": "✅ Completed", "Commit": "69b5a8b", "Notes": "Stacked bar, Pareto scatter, priority bar"},
    # Day 7 tasks
    {"Task ID": "D7-1", "Description": "src/report_builder.py — load_kpis(), build_latex(), compile_pdf(), run()", "Status": "✅ Completed", "Commit": "Day 7 (this session)", "Notes": "12-page PDF at report/report_draft_v1.pdf"},
    {"Task ID": "D7-2", "Description": "run_all.sh — 10-stage one-shot pipeline script (bash run_all.sh --check)", "Status": "✅ Completed", "Commit": "Day 7 (this session)", "Notes": "Set -euo pipefail; stages 1–10"},
    {"Task ID": "D7-3", "Description": "notebooks/pritam_temp_notebooks/day6_7_session_summary.ipynb", "Status": "✅ Completed", "Commit": "Day 7 (this session)", "Notes": "This notebook"},
]

df_tasks = pd.DataFrame(tasks)
display(df_tasks.style.set_caption("Day 6 & 7 Task Tracker"))

---
## Section 2 — Day 6 Completed Tasks

### What was built

**`src/kpi_reporter.py`** (new file)
- `load_forward(path)` / `load_reverse(path)` / `load_hybrid(path)` — load and column-prefix each KPI CSV
- `build_combined(fwd, rev, hybrid)` — merge on `zone_id`, compute `separate_cost_R$` and `combined_vehicles`
- `zone_priority_ranking(df)` — rank zones by saving (SDVRP) or separate cost (fallback)
- `print_summary(df)` — formatted console table
- `run(...)` → `outputs/combined_kpi_report.csv`

**`src/joint_optimizer.py`** — `pareto_sweep()` appended
- 5×5 α/β grid (γ=δ=(1−α−β)/2), up to 15 valid combos
- Non-dominance filter on (routing cost, T_pen)
- Knee point = min-normalised distance to ideal (0,0)
- Saves `outputs/pareto_results.csv`

**`notebooks/06_kpi_and_pareto.ipynb`** — 12 cells
- Header, KPI theory, imports, kpi_reporter call, stacked bar chart
- Pareto math, pareto_sweep call, scatter chart, priority ranking, checklist

In [ ]:
from pathlib import Path

day6_outputs = {
    "src/kpi_reporter.py":               "src/kpi_reporter.py",
    "outputs/combined_kpi_report.csv":   "outputs/combined_kpi_report.csv",
    "outputs/pareto_results.csv":        "outputs/pareto_results.csv",
    "notebooks/06_kpi_and_pareto.ipynb": "notebooks/06_kpi_and_pareto.ipynb",
}

ROOT = Path(".")
print("Day 6 artefact verification:")
all_ok = True
for label, rel in day6_outputs.items():
    p = ROOT / rel
    status = "✅" if p.exists() else "❌ MISSING"
    size = f"  ({p.stat().st_size:,} bytes)" if p.exists() else ""
    print(f"  {status}  {label}{size}")
    if not p.exists():
        all_ok = False
print()
print("All Day 6 artefacts present:", "YES ✅" if all_ok else "NO ❌")

---
## Section 3 — Day 7 Task: LaTeX Report Generation

### `src/report_builder.py` — design

```
load_kpis(outputs_dir, data_dir)
    → reads forward/reverse/combined KPI CSVs, clf JSON, joint JSON, etc.
    → returns dict of DataFrames + plain values

build_latex(kpis, figures_dir)
    → assembles 10-section LaTeX document as a Python string
    → includes optional figure blocks (_maybe_fig) — skipped if PNG absent
    → KPI tables generated from live DataFrames via _fwd_kpi_table / _rev_kpi_table

compile_pdf(tex_path)
    → runs pdflatex twice (for ToC + cross-references)
    → cwd = project root so relative image paths resolve
    → returns Path to PDF; raises RuntimeError only if no PDF produced

run(output_dir, outputs_dir, data_dir, compile=True)
    → convenience wrapper: load → build → write .tex → compile .pdf
```

### Report structure (10 sections, 12 pages)

| Section | Content |
|---------|---------|
| 1 | Introduction — research question |
| 2 | Dataset & Environment (Olist SP, uv, OR-Tools) |
| 3 | Stage 1 — Dark Store Placement (K-Means + p-Median, K=11) |
| 4 | Stage 2 — Forward VRP CVRPTW (1,069 km, −98.58%) |
| 5 | Stage 3 — Return Classifier (XGBoost, ROC-AUC=0.897) |
| 6 | Stage 4 — Reverse VRP (946 km, R$2,169) |
| 7 | Stage 5 — Joint MILP Optimiser (Z=54.38 Optimal) |
| 8 | Stage 6 — SDVRP Hybrid Routing |
| 9 | Stage 7 — Combined KPI + Pareto Sweep |
| 10 | Summary Table + Reproducibility + References |

In [ ]:
import sys
sys.path.insert(0, '.')

from src.report_builder import load_kpis, build_latex

kpis = load_kpis(outputs_dir="outputs", data_dir="data")
tex_str = build_latex(kpis, figures_dir="outputs")
print(f"\nLaTeX source length: {len(tex_str):,} characters")

# Show first 60 lines of generated LaTeX
lines = tex_str.splitlines()
print(f"First 60 lines of report_draft_v1.tex:")
print("-" * 60)
for line in lines[:60]:
    print(line)
print("...")

---
## Section 4 — Day 7 Task: PDF Publishing

In [ ]:
from src.report_builder import run as build_report
from pathlib import Path

pdf_path = build_report(output_dir="report", outputs_dir="outputs", data_dir="data")

# Verify
pdf = Path("report/report_draft_v1.pdf")
tex = Path("report/report_draft_v1.tex")

print("\nDay 7 report artefacts:")
for p in [tex, pdf]:
    status = "✅" if p.exists() else "❌ MISSING"
    size = f"  ({p.stat().st_size:,} bytes)" if p.exists() else ""
    print(f"  {status}  {p}{size}")

---
## Section 5 — Remaining Tasks

The table below lists items still pending after Day 6 & 7.

In [ ]:
remaining = [
    {
        "Task ID": "D8-1",
        "Description": "Run notebooks/06_kpi_and_pareto.ipynb (cells not yet executed — chart PNGs missing)",
        "Priority": "High",
        "Effort": "~10 min",
        "Notes": "Generates combined_cost_by_zone.png, pareto_tradeoff.png, sdvrp_priority_ranking.png for report"
    },
    {
        "Task ID": "D8-2",
        "Description": "Re-run report_builder.py after notebook 06 charts are generated (figures currently missing → draft placeholders)",
        "Priority": "High",
        "Effort": "~5 min",
        "Notes": "PDF will include all charts once PNGs exist"
    },
    {
        "Task ID": "D8-3",
        "Description": "Run all_zones_aggregator.py to produce hybrid_kpi_summary.csv and hybrid_routes.json",
        "Priority": "Medium",
        "Effort": "~30 min",
        "Notes": "SDVRP hybrid routes for all 11 zones; updates combined_kpi_report with saving_R$ column"
    },
    {
        "Task ID": "D8-4",
        "Description": "Final end-to-end run: bash run_all.sh (full pipeline, raw data required)",
        "Priority": "Low",
        "Effort": "~2–3 h",
        "Notes": "Requires raw Olist CSVs in data/raw/"
    },
    {
        "Task ID": "D8-5",
        "Description": "Team PR review: merge teammate branches (Sneha/Vybhav/Pranav) and re-run integration tests",
        "Priority": "Medium",
        "Effort": "~1 h",
        "Notes": "PRs #31/#32 already merged; check for Day 7 PRs"
    },
    {
        "Task ID": "D8-6",
        "Description": "Finalise report — add abstract, executive summary, polish figure captions",
        "Priority": "Medium",
        "Effort": "~1–2 h",
        "Notes": "After charts integrated; target: report/report_final_v2.pdf"
    },
]

df_rem = pd.DataFrame(remaining)
display(df_rem.style.set_caption("Remaining Tasks — Day 8 Backlog"))
print(f"\nTotal remaining items: {len(remaining)}")

---
## Section 6 — Day 7 Bug Fix: Pareto Sweep Single-Point Issue

### Root Causes (3 bugs compounded)

| # | Location | Bug | Effect |
|---|----------|-----|--------|
| 1 | Notebook cell 8 | `return_probs = pd.Series(dtype=float)` (empty) | `expected_returns = 0` → `T_pen = 0` always |
| 2 | `pareto_sweep()` | `gamma = delta = (1−α−β)/2` made delta tiny | MILP activated min vehicles for every weight combo |
| 3 | `extract_results()` | Returned raw `expected_returns` constant as T_pen | T_pen same across all 15 rows regardless of active vehicles |

### Fix: ε-constraint enumeration

```
OLD: run MILP for each (α, β) pair  →  same vehicles activated every time
NEW: enumerate all (k_fwd, k_rev) in {1..n_fwd} × {1..n_rev} analytically
     cheapest k vehicles per type; T_pen = γ × E × (1 − k_rev / n_rev)
```

Result: **2 genuine Pareto-optimal points** on (routing cost, T\_pen) axes:

| Point | k\_fwd | k\_rev | C\_routing | T\_pen | Meaning |
|-------|--------|--------|------------|--------|----------|
| A | 1 | 1 | R\$120.80 | 7.074 | Min cost; accept late-return penalty |
| B (knee) | 1 | 2 | R\$214.52 | 0.000 | Zero penalty; deploy both reverse vehicles |


In [ ]:
import sys
sys.path.insert(0, '.')
import numpy as np
import pandas as pd
from src.joint_optimizer import pareto_sweep

fwd_routes = pd.read_csv('outputs/forward_routes.csv')
rev_routes  = pd.read_csv('outputs/reverse_routes.csv')

rng = np.random.default_rng(42)
return_probs = pd.Series(rng.beta(2, 15, size=fwd_routes['vehicle_id'].nunique()*10).astype(float))

pareto_df = pareto_sweep(fwd_routes, rev_routes, return_probs,
                         output_path='outputs/pareto_results.csv')

print('\nPareto results (corrected):')
display(pareto_df[['n_fwd_active','n_rev_active','C_fwd','C_rev','T_pen',
                   'total_routing_cost','is_pareto','is_knee']])


---
## Day 6 & 7 EOD Checklist

| Task | Status |
|------|--------|
| `src/kpi_reporter.py` created and smoke-tested | ✅ |
| `pareto_sweep()` added to `src/joint_optimizer.py` | ✅ |
| `notebooks/06_kpi_and_pareto.ipynb` created (12 cells) | ✅ |
| `src/report_builder.py` created | ✅ |
| `report/report_draft_v1.tex` generated | ✅ |
| `report/report_draft_v1.pdf` compiled (12 pages) | ✅ |
| `run_all.sh` created with 10-stage pipeline | ✅ |
| Session summary notebook created | ✅ |
| **Bug fix**: Pareto sweep ε-constraint redesign (commit `9766c07`) | ✅ |
| Report regenerated post Pareto fix | ✅ |
| Day 7 all changes committed and pushed to `pritam_temp_apr5` | ✅ |

**Key metrics confirmed:**
- Forward VRP: 1,069.59 km · R$2,704.40 · 22 vehicles · **98.58% improvement**
- Reverse VRP: 946.36 km · R$2,169.51 · 15 vehicles · 576 pickups
- Return classifier: ROC-AUC = **0.8969** (target ≥ 0.70 ✅)
- Joint MILP: Z = **54.38 Optimal**
- Separate baseline total: **R$4,873.91**
- Zone 8: Rank 1 priority (R$570.84 combined cost)
- Pareto knee: k\_fwd=1, k\_rev=2 (zero T\_pen, R$214.52 routing cost)
